# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zayer1/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

* **Unit of analysis:** One row = one content item (page) per client on a specific day. (Grain: `report_date` × `client_hash_id` × `content_hash_id`)
* **Time window:** The month of March 2026 (`month=2026-03`), which acts as our mid-panel iteration window. I will also make sure to use per-client windows based on `dim_clients.gsc_data_start` instead of a global calendar window.


In [1]:
import os
import duckdb
import pandas as pd

# Connect and configure Hugging Face token
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

print("Setup complete. Token loaded from environment.")


Setup complete. Token loaded from environment.


## 2. Fields: feature / label / context / excluded

* **Feature:** `content_age_days`, `word_count`, `impressions_prev_30d`, `has_ai_sessions`. (These are safely knowable *before* the prediction window).
* **Label / proxy:** `is_declining_label` (calculated from `trend_direction` which looks at last 30d vs prev 30d). `trend_pct` and `trend_direction` also go here.
* **Context:** `content_hash_id`, `client_hash_id`, `report_date`, `content_type`. Used for joining, splits, and grouping.
* **Excluded:** Any rows where `ga4_data_available IS NOT TRUE` or `gsc_data_available IS NOT TRUE` because they contain structural zeros instead of real absence of activity. I also exclude the final month (June 2026) to prevent test leakage.


In [2]:
# Code cell left intentionally blank for this section as verification is below.
pass


## 3. Verify it with queries (grain, counts, missing values, windows)

**Five Features used for the modeling trap:**
1. `content_age_days`: Knowable at the decision moment because the creation date is static.
2. `word_count`: Knowable because it is a property of the content measured at that time.
3. `impressions_prev_30d`: Knowable because it strictly looks at the window *before* the prediction period.
4. `clicks_prev_30d`: Knowable because it strictly captures historical clicks.
5. `has_ai_sessions`: Knowable from past GA4 traffic logs prior to the decision point.


In [3]:
# 1. Grain check: report_date x client_hash_id x content_hash_id should be unique
q_grain = """
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
"""
print("Grain check (should be empty if grain holds):")
print(con.execute(q_grain).df())
print("-" * 50)

# 2. Row counts and date spans
q_counts = """
SELECT 
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date,
    COUNT(DISTINCT client_hash_id) as distinct_clients
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
print("Row counts and dates for our slice:")
print(con.execute(q_counts).df())
print("-" * 50)

# 3. Availability filter check
q_avail = """
SELECT 
    COUNT(*) as total_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as valid_ga4_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*)::FLOAT * 100 as pct_valid
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
"""
print("Availability filter (ga4_data_available IS TRUE):")
print(con.execute(q_avail).df())


Grain check (should be empty if grain holds):


Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []
--------------------------------------------------
Row counts and dates for our slice:


   total_rows start_date   end_date  distinct_clients
0     9841378 2026-03-01 2026-03-31                55
--------------------------------------------------
Availability filter (ga4_data_available IS TRUE):


   total_rows  valid_ga4_rows  pct_valid
0     9841378        413966.0   4.206382


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 4. The Trap (Leakage)
q_trap = """
SELECT 
    gsc_impressions, 
    sessions_ai, 
    ga4_data_available,
    CASE WHEN gsc_impressions > 10 THEN 1 ELSE 0 END as high_traffic_label,
    CASE WHEN gsc_impressions > 10 THEN 'high' ELSE 'low' END as leaky_category
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE ga4_data_available IS TRUE 
LIMIT 50000
"""
df = con.execute(q_trap).df().dropna()

# Convert categorical to numeric for sklearn
df['leaky_feature'] = df['leaky_category'].map({'high': 1, 'low': 0})

features = ['sessions_ai']
leaky_features = features + ['leaky_feature']

X_train_h, X_test_h, y_train, y_test = train_test_split(df[features], df['high_traffic_label'], test_size=0.2, random_state=42)
X_train_l, X_test_l, _, _ = train_test_split(df[leaky_features], df['high_traffic_label'], test_size=0.2, random_state=42)

# Honest model
rf_honest = RandomForestClassifier(n_estimators=10, random_state=42, max_depth=5).fit(X_train_h, y_train)
acc_honest = accuracy_score(y_test, rf_honest.predict(X_test_h))

# Leaky model
rf_leaky = RandomForestClassifier(n_estimators=10, random_state=42, max_depth=5).fit(X_train_l, y_train)
acc_leaky = accuracy_score(y_test, rf_leaky.predict(X_test_l))

print(f"Honest accuracy: {acc_honest:.2%}")
print(f"Leaky accuracy (with trend_pct trap): {acc_leaky:.2%} -> PERFECT score because we leaked the label source!")


Honest accuracy: 77.20%
Leaky accuracy (with trend_pct trap): 100.00% -> PERFECT score because we leaked the label source!


## 4. Data limits

* **Unbalanced panel:** Different clients have vastly different history depths (check `gsc_data_start`), meaning we cannot train assuming every client has identical seasonal coverage.
* **Leakage overlap:** The daily performance table is limited. We need to be careful with features overlapping with labels.


# No code needed here
pass

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
